# Scraping for Urban Farms in Dallas County 

Things to do: 
- Get rid of farms that aren't in Dallas County 

In [1]:
import requests
import pandas as pd 
import geopandas as gpd 
from sklearn.cluster import DBSCAN # for dropping fuzzy duplicates 

The Dallas Coalition for Hunger Solutions hosts a [map](https://www.dallashunger.org/garden-locator-map) of urban farms, including community gardens, school food gardens, urban farms, market gardens, and teaching gardens. It is based on self-reporting through a form on the page, so it is likely not comprehensive (it doesn't even include Joppy Momma's Farm), but it's  an excellent starting point. 

In [2]:
url = 'https://core.service.elfsight.com/p/boot/?page=https%3A%2F%2Fwww.dallashunger.org%2Fgarden-locator-map&w=5145b646-fd4c-4aa9-b6cf-d7e8b826e0ad'
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
raw_data = response.json()

# 1. Safely extract the widgets container
widgets_container = raw_data['data']['widgets']

# 2. Extract the first widget payload regardless of whether it is a list or a dict
if isinstance(widgets_container, dict):
    # If it's a dict, get the first value safely without using a key like 0
    first_widget_config = next(iter(widgets_container.values()))
elif isinstance(widgets_container, list):
    # If it's a list, look at the first element safely
    first_widget_config = widgets_container[0]
else:
    raise TypeError(f"Unexpected type for widgets: {type(widgets_container)}")

# 3. Drill down into the markers
markers = first_widget_config['data']['settings']['markers']

# 4. Check if markers is a list or a dictionary
if isinstance(markers, dict):
    # If markers is a dictionary, use from_dict orient='index'
    df = pd.DataFrame.from_dict(markers, orient='index')
elif isinstance(markers, list):
    # If markers is already a clean list of dictionaries, load it directly
    df = pd.DataFrame(markers)
else:
    raise TypeError(f"Unexpected type for markers: {type(markers)}")

print(f"Success! Extracted {len(df)} urban farms.")

# 5. Filter for just the essential spatial and descriptive data
relevant_cols = [col for col in ['title', 'address', 'lat', 'lng', 'categoryId'] if col in df.columns]

if relevant_cols:
    clean_df = df[relevant_cols]
    print("\nCleaned Data Sample:")
    print(clean_df.head())
    clean_df.to_csv("dallas_urban_farms.csv", index=False)
else:
    print("\nColumns found in the dataframe:")
    print(df.columns.tolist())
    print("\nFirst row sample:")
    print(df.iloc[0].to_dict())

Success! Extracted 191 urban farms.

Columns found in the dataframe:
['position', 'coordinates', 'icon', 'category', 'linkUrl', 'infoWindow', 'infoTitle', 'infoDescription', 'infoImage', 'infoAddress', 'infoSite', 'infoPhone', 'infoEmail', 'infoWorkingHours', 'infoWindowOpenedByDefault', 'markerClickAction', 'animation', 'id', 'chosen', 'selected']

First row sample:
{'position': '1513 S. Beltline Rd., Grand Prairie, TX 75052', 'coordinates': '32.7239892, -96.9960085', 'icon': None, 'category': 'afb9d952-8c58-405a-9ccc-cc944ec6ad6c', 'linkUrl': '', 'infoWindow': True, 'infoTitle': 'The Haven Community Garden', 'infoDescription': '<div>38 raised beds; 8 garden rows; 14 fruit trees; hours of operation: dawn till dusk; no annual fee; funded by Keep Grand Prairie Beautiful.</div>', 'infoImage': None, 'infoAddress': '', 'infoSite': '', 'infoPhone': '', 'infoEmail': ' sharon.dehnert@tx.rr.com', 'infoWorkingHours': '', 'infoWindowOpenedByDefault': False, 'markerClickAction': 'infoWindow', 'an

In [4]:
farms_clean_df = df[['position', 'coordinates', 'category', 'infoTitle', 'infoEmail']]
farms_clean_df.to_csv("dallasHungerDirectory.csv", index=False)

In [8]:
# 1. Set your endpoint (Make sure it ends in /query)
url = 'https://services2.arcgis.com/rwnOSbfKSwyTBcwN/arcgis/rest/services/DallasUrbanGarden2024_Geocoded/FeatureServer/1/query'

# 2. Setup the loop variables
all_features = []
offset = 0
chunk_size = 1000 # A safe number that won't trigger server limits
has_more_data = True

print("Pinging the ArcGIS server...")

while has_more_data:
    # Define the query parameters
    params = {
        "where": "1=1",
        "outFields": "*",
        "returnGeometry": "true",
        "f": "geojson",
        "resultOffset": offset,
        "resultRecordCount": chunk_size
    }
    
    # Execute the request
    response = requests.get(url, params=params)
    response.raise_for_status() # Catch any HTTP errors immediately
    
    data = response.json()
    
    # Extract the chunk of data
    features = data.get("features", [])
    all_features.extend(features)
    
    # Check if we hit the end of the dataset
    # Esri often includes an "exceededTransferLimit" flag if there is more data waiting
    if data.get("exceededTransferLimit") or len(features) == chunk_size:
        offset += chunk_size
        print(f"Downloaded {len(all_features)} records so far...")
    else:
        has_more_data = False # We've reached the end

print(f"Extraction complete! Total records: {len(all_features)}")

# 3. Convert directly into a GeoDataFrame
# Because we requested standard GeoJSON, Geopandas can ingest the raw dictionary list directly
if all_features:
    # Reconstruct a valid GeoJSON object in memory
    geojson_payload = {
        "type": "FeatureCollection",
        "features": all_features
    }
    
    gdf = gpd.GeoDataFrame.from_features(geojson_payload["features"])
    
    # Set the default coordinate reference system to standard GPS (WGS84)
    gdf.set_crs(epsg=4326, inplace=True)
    
    print("\nSample of extracted data:")
    print(gdf.head())

Pinging the ArcGIS server...
Extraction complete! Total records: 59

Sample of extracted data:
                     geometry  ObjectID        Loc_name USER_City  \
0  POINT (-96.80498 32.74688)         1  CrmDallasStree    Dallas   
1  POINT (-96.86937 32.85665)         2  CrmDallasStree    Dallas   
2  POINT (-96.75866 32.77632)         3  CrmDallasStree    Dallas   
3   POINT (-96.7537 32.73491)         4  CrmDallasStree    Dallas   
4  POINT (-96.69398 32.84711)         5  CrmDallasStree    Dallas   

  USER_Subregion                                      USER_SiteName  \
0  Dallas County                            10 St. Community Garden   
1  Dallas County         Bachman Lake Together Family Center Garden   
2  Dallas County                                Big Tex Urban Farms   
3  Dallas County                                       Bonton Farms   
4  Dallas County  Central Lutheran Garden, at Central Lutheran C...   

   USER_OldId                                   USER_ARC_Addres

In [9]:
gdf.to_file("dallasClimateDirectory24", driver="GeoJSON")

In [10]:
# 1. Set your endpoint (Make sure it ends in /query)
url = 'https://services2.arcgis.com/rwnOSbfKSwyTBcwN/arcgis/rest/services/UrbanGardensNov2022_Geocoded/FeatureServer/0/query'

# 2. Setup the loop variables
all_features = []
offset = 0
chunk_size = 1000 # A safe number that won't trigger server limits
has_more_data = True

print("Pinging the ArcGIS server...")

while has_more_data:
    # Define the query parameters
    params = {
        "where": "1=1",
        "outFields": "*",
        "returnGeometry": "true",
        "f": "geojson",
        "resultOffset": offset,
        "resultRecordCount": chunk_size
    }
    
    # Execute the request
    response = requests.get(url, params=params)
    response.raise_for_status() # Catch any HTTP errors immediately
    
    data = response.json()
    
    # Extract the chunk of data
    features = data.get("features", [])
    all_features.extend(features)
    
    # Check if we hit the end of the dataset
    # Esri often includes an "exceededTransferLimit" flag if there is more data waiting
    if data.get("exceededTransferLimit") or len(features) == chunk_size:
        offset += chunk_size
        print(f"Downloaded {len(all_features)} records so far...")
    else:
        has_more_data = False # We've reached the end

print(f"Extraction complete! Total records: {len(all_features)}")

# 3. Convert directly into a GeoDataFrame
# Because we requested standard GeoJSON, Geopandas can ingest the raw dictionary list directly
if all_features:
    # Reconstruct a valid GeoJSON object in memory
    geojson_payload = {
        "type": "FeatureCollection",
        "features": all_features
    }
    
    gdf = gpd.GeoDataFrame.from_features(geojson_payload["features"])
    
    # Set the default coordinate reference system to standard GPS (WGS84)
    gdf.set_crs(epsg=4326, inplace=True)
    
    print("\nSample of extracted data:")
    print(gdf.head())

Pinging the ArcGIS server...
Extraction complete! Total records: 54

Sample of extracted data:
                     geometry  OBJECTID  Score  \
0  POINT (-96.76109 32.87116)         1  100.0   
1  POINT (-96.75054 32.86678)         2  100.0   
2   POINT (-96.7709 32.77334)         3  100.0   
3  POINT (-96.78581 32.78083)         4  100.0   
4  POINT (-96.73952 32.78156)         5  100.0   

                               Match_addr  \
0             8361 PARK LN, DALLAS, 75231   
1        6723 EASTRIDGE DR, DALLAS, 75231   
2         2641 JEFFRIES ST, DALLAS, 75215   
3  458 S GOOD LATIMER EXPY, DALLAS, 75201   
4          4815 SILVER AVE, DALLAS, 75223   

                                LongLabel               ShortLabel  \
0             8361 PARK LN, DALLAS, 75231             8361 PARK LN   
1        6723 EASTRIDGE DR, DALLAS, 75231        6723 EASTRIDGE DR   
2         2641 JEFFRIES ST, DALLAS, 75215         2641 JEFFRIES ST   
3  458 S GOOD LATIMER EXPY, DALLAS, 75201  458 S GOOD

In [11]:
gdf.to_file('dallasClimateDirectory22', driver='GeoJSON')

In [2]:
d24 = gpd.read_file('dallasClimateDirectory24')
d22 = gpd.read_file('dallasClimateDirectory22')

In [3]:
if d24.crs != d22.crs:
    print("CRS mismatch detected. Aligning projections...")
    d22 = d22.to_crs(d24.crs)

climateDirect = pd.concat([d24, d22], ignore_index=True)

climateDirectGDF = gpd.GeoDataFrame(climateDirect, geometry='geometry', crs=d24.crs)

In [4]:
climateDirectGDF = climateDirectGDF.iloc[:, 0:62]
climateDirectGDF = climateDirectGDF.iloc[:, list(range(0,8)) + list(range(14, 33)) + list(range(59, 62))]

In [5]:
climateDirectGDF = climateDirectGDF.drop_duplicates(subset=['geometry'], ignore_index=True)

In [6]:
dHunger = pd.read_csv('dallasHungerDirectory.csv')
split_coords = dHunger['coordinates'].str.split(',', expand=True)

In [7]:
dHungerGDF = gpd.GeoDataFrame(
    dHunger,
    geometry=gpd.points_from_xy(split_coords[1].astype(float), split_coords[0].astype(float)),
    crs="EPSG:4326"
)

In [8]:
if climateDirectGDF.crs != dHungerGDF.crs:
    print("CRS mismatch detected. Aligning projections...")
    climateDirectGDF = climateDirectGDF.to_crs(dHungerGDF.crs)

dallasFarmsDF = pd.concat([climateDirectGDF, dHungerGDF], ignore_index=True)

dallasFarmsGDF = gpd.GeoDataFrame(dallasFarmsDF, geometry='geometry', crs=dHungerGDF.crs)

In [9]:
dallasFarmsGDF = dallasFarmsGDF.drop_duplicates(subset=['geometry'], ignore_index=True)

In [31]:
dallasFarmsGDF.info()

USER_SiteName       183
USER_ARC_Address    231
geometry              0
position            107
infoTitle           107
SiteName              0
dtype: int64

In [10]:
dallasFarmsGDF = dallasFarmsGDF[['USER_SiteName', 'USER_ARC_Address', 'geometry', 'position', 'infoTitle']]

In [11]:
# Find rows where BOTH columns are NOT null
conflicts = dallasFarmsGDF[dallasFarmsGDF['USER_SiteName'].notna() & dallasFarmsGDF['infoTitle'].notna()]

print(f"Number of conflicting rows: {len(conflicts)}")

Number of conflicting rows: 0


In [12]:
dallasFarmsGDF['SiteName'] = dallasFarmsGDF['USER_SiteName'].combine_first(dallasFarmsGDF['infoTitle'])
dallasFarmsGDF['Address'] = dallasFarmsGDF['USER_ARC_Address'].combine_first(dallasFarmsGDF['position'])
dallasFarmsGDF = dallasFarmsGDF[['SiteName', 'Address', 'geometry']]

In [38]:
dallasFarmsGDF.to_file('dallasFarmsClean.geojson', driver='GeoJSON')

In [13]:
dallasFarmsGDF.sort_values(by='SiteName')

,SiteName,Address,geometry
0,10 St. Community Garden,"1300 East Clarendon Dr., Dallas, TX, 75203",POINT (-96.80498 32.74688)
113,10 St. Community Garden,"1300 E. Clarendon Dr., Dallas, TX, 75203",POINT (-96.80502 32.74701)
94,10 St. Community Garden,NaN,POINT (-96.8053 32.74687)
114,A Special Blend,"2378 E. Ledbetter Dr., Dallas, TX, 75216",POINT (-96.78242 32.68898)
115,Adamson High School Garden,"309 E 9th St., Dallas, TX, 75203",POINT (-96.82019 32.74824)
...,...,...,...
58,Worth Street Community Garden,"513 N Peak St, Dallas, Texas, 75246",POINT (-96.77293 32.79315)
262,Worth Street Community Garden,"517 N. Peak St., Dallas, TX, 75046",POINT (-96.77315 32.79313)
83,Worth Street Garden,NaN,POINT (-96.77258 32.79299)
273,Wyatt-Shockley Community Garden,"3610 Dunbar Street, Dallas, Texas 76215",POINT (-96.75996 32.76743)


In [41]:
# 1. Project your GeoDataFrame to a CRS that uses meters (so we can measure actual distance)
# EPSG:3857 (Web Mercator) is a safe default if you aren't sure which local CRS to use
gdf_meters = dallasFarmsGDF.to_crs(epsg=3857)

# 2. Extract the raw X and Y coordinates into a format scikit-learn can read
coords = gdf_meters.get_coordinates()

# 3. Set up the clustering algorithm
# eps=50 means group points that are within 50 meters of each other
# min_samples=1 ensures unique points without duplicates get their own cluster, rather than being marked as noise
clustering = DBSCAN(eps=50, min_samples=1).fit(coords)

# 4. Assign the resulting cluster IDs back to your original GeoDataFrame
dallasFarmsGDF['cluster_id'] = clustering.labels_

# 5. Drop duplicates based on the shared cluster ID
dallasFarmsCleanGDF = gdf.drop_duplicates(subset=['cluster_id'], keep='first')

# 6. Clean up
dallasFarmsCleanGDF = dallasFarmsCleanGDF.drop(columns=['cluster_id'])

ValueError: Input X contains NaN.
DBSCAN does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [46]:
gdf_meters['geometry'].isna().sum()

np.int64(0)

In [48]:
# This specifically looks for empty shapes, not just null rows
print(gdf_meters.geometry.is_empty.sum())

0


In [53]:
# Look at the minimum and maximum Y (latitude) values to spot typos
print(dallasFarmsGDF.geometry.y.min(), dallasFarmsGDF.geometry.y.max())

32.5555583 33.6551999


In [51]:
max_lat = dallasFarmsGDF.geometry.y.max()

# 2. Filter the dataframe to show only the row(s) with that exact coordinate
bad_row = dallasFarmsGDF[dallasFarmsGDF.geometry.y == max_lat]

# 3. View the result
print(bad_row)

                       SiteName                            Address  \
141  DC Homestead Farm to Table  4913 N. FM 1752, Savoy, TX, 75479   

                      geometry  
141  POINT (-96.35699 33.6552)  


In [60]:
# Check the dataframe AFTER the CRS projection, but BEFORE get_coordinates()
print(gdf_meters[gdf_meters.geometry.isna() | gdf_meters.geometry.is_empty])

Empty GeoDataFrame
Columns: [SiteName, Address, geometry]
Index: []


In [61]:
# Look specifically for geometries where the X coordinate is NaN
ghost_rows = gdf_meters[gdf_meters.geometry.x.isna()]

print(ghost_rows)

                           SiteName                                 Address  \
199  Mohawk Community Garden (RISD)  1500 Mimosa Dr., Richardson, TX, 75080   

            geometry  
199  POINT (NaN NaN)  


In [68]:
dallasFarmsGDF['geometry'].info()

<class 'geopandas.geoseries.GeoSeries'>
RangeIndex: 290 entries, 0 to 289
Series name: geometry
Non-Null Count  Dtype   
--------------  -----   
290 non-null    geometry
dtypes: geometry(1)
memory usage: 2.4 KB


In [14]:
from shapely.geometry import Point

# 1. Identify the index label of the row you want to fix
row_index = 199 # Replace with the actual index number of the broken row

# 2. Create the new, correct geometry (Remember: Longitude first!)
correct_point = Point(-96.76281377866194, 32.97141589057731)

# 3. Use .at to surgically replace the geometry in that exact cell
dallasFarmsGDF.at[row_index, 'geometry'] = correct_point

In [15]:
# 1. Project to meters
gdf_meters = dallasFarmsGDF.to_crs(epsg=3857)

# 2. THE ULTIMATE FIX: Filter out missing, empty, AND ghost coordinates
gdf_meters_valid = gdf_meters[
    ~gdf_meters.geometry.isna() & 
    ~gdf_meters.geometry.is_empty & 
    ~gdf_meters.geometry.x.isna()  # Catches the POINT (nan nan)
].copy()

# 3. Extract the clean coordinates
coords = gdf_meters_valid.get_coordinates()

# 4. Run DBSCAN
clustering = DBSCAN(eps=100, min_samples=1).fit(coords)

# 4. Assign the resulting cluster IDs back to your original GeoDataFrame
dallasFarmsGDF['cluster_id'] = clustering.labels_

# 5. Drop duplicates based on the shared cluster ID
dallasFarmsCleanGDF = dallasFarmsGDF.drop_duplicates(subset=['cluster_id'], keep='first')

# 6. Clean up
dallasFarmsCleanGDF = dallasFarmsCleanGDF.drop(columns=['cluster_id'])

In [16]:
dallasFarmsCleanGDF.to_file('dallasFarmsClean.geojson', driver='GeoJSON')

In [ ]:
dallasFarmsCleanGDF.info(